# Session 3 — Data Quality Assessment
**Pandas & Data Manipulation** · Albert School × Politecnico di Milano

| Part | What you do | Time |
|---|---|---|
| **1. Demo** | Run each cell: the examples of the slides, on the tiny table | 10 min |
| **2. Practice** | 10 questions on real taxi data, from ★ to ★★★ | 60 min |

**Today we assess, we do not clean.** Every check returns a True/False **mask**:
`mask.sum()` → how many rows · `mask.mean()` → which share.

| Dimension | Question | Main tools |
|---|---|---|
| 🟦 Missing values | Is the value there? | `isna()`, `value_counts()` |
| 🟧 Inconsistencies | Is it written one way? | `unique()`, `nunique()`, `.str`, `pd.to_numeric()` |
| 🟩 Duplication | Is each fact recorded once? | `duplicated()`, `subset=`, `keep=` |
| 🟨 Accuracy | Is the value plausible? | `describe()`, `between()`, `isin()`, `quantile()` |

## Part 1 — Demo
### 0 · The tiny table from the slides
It hides all four problems: missing cells, a placeholder (`"unknown"`), two spellings of a country,
a duplicated order, an impossible age and a negative total.

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 101, 105],
    "customer": ["Ana", "Ben", None, "Dan", "Ana", "Eva"],
    "country":  ["FR", "it", "IT", "FR", "FR", " ES"],
    "age":      [34, None, 29, 230, 34, 41],
    "total":    [120, 80, -50, 95, 120, None],
    "status":   ["shipped", "delivered", "shipped", "unknown", "shipped", "returned"],
})
df

### 1 · Missing values
**NaN** (*Not a Number*) marks an empty cell. Statistics skip it silently, and it is never equal to anything.

In [ ]:
s = pd.Series([10, None, 30])
print(s.tolist(), s.dtype)          # None became NaN, the integers became floats
print(len(s), s.count())            # 3 values, 2 not missing
print(s.mean())                     # 20.0 = (10 + 30) / 2: NaN is skipped
print(np.nan == np.nan)             # False -> never test "== np.nan", use isna()

`df.isna()` → a True/False map of the whole table (True = missing). `df.notna()` is the opposite.

In [ ]:
df.isna()

True counts as 1 and False as 0: `sum()` counts the missing cells, `mean()` gives the share.

In [ ]:
pd.DataFrame({
    "n_missing": df.isna().sum(),          # how many per column
    "pct": df.isna().mean() * 100,         # which share per column (%)
}).round(1)

`axis=1` works across each row: `any(axis=1)` is True when a row has at least one NaN.

In [ ]:
incomplete = df.isna().any(axis=1)
print(incomplete.sum(), "incomplete rows")
df[incomplete]

In [ ]:
df.info()      # the Non-Null Count column gives the same information

**Placeholders** mean "we do not know" but are stored as text or numbers: `isna()` does not see them.

In [ ]:
print(df["status"].isna().sum())     # 0 ...
df["status"].value_counts()          # ... but "unknown" is a missing value in disguise

`read_csv` turns empty cells, `N/A`, `NA`, `null` into NaN by default. Declare other placeholders with `na_values=`.

In [ ]:
from io import StringIO

csv = "city,temp\nParis,12\nRome,N/A\nMilan,unknown\nLyon,\n"
d1 = pd.read_csv(StringIO(csv))
d2 = pd.read_csv(StringIO(csv), na_values=["unknown"])
print("default   :", d1["temp"].isna().sum(), "missing, dtype", d1["temp"].dtype)
print("na_values :", d2["temp"].isna().sum(), "missing, dtype", d2["temp"].dtype)

### 2 · Inconsistencies
`unique()` lists the distinct values, `nunique()` counts them, `value_counts()` gives each value with its frequency.

In [ ]:
print(df["country"].unique())
print(df["country"].nunique(), "different strings")
df["country"].value_counts()

Normalise a **copy** (`.str.strip()` removes spaces, `.str.upper()` sets one case) to measure the variants. We do **not** write it back into `df`.

In [ ]:
norm = df["country"].str.strip().str.upper()
print(norm.nunique(), "countries after normalising")
print((df["country"] != norm).sum(), "rows written differently")
pd.DataFrame({"country": df["country"], "normalised": norm, "differs": df["country"] != norm})

A numeric column stored as text hides at least one non-number. `pd.to_numeric(errors="coerce")` turns non-numbers into NaN.

In [ ]:
p = pd.Series(["10", "12.5", "n.d.", "8"])
num = pd.to_numeric(p, errors="coerce")
print(p.dtype, "->", num.dtype)
print(num.isna().sum(), "value is not a number:", p[num.isna()].tolist())

Related columns must agree: here `total` should be `qty * price`.

In [ ]:
o = pd.DataFrame({"qty": [2, 1, 3, 2],
                  "price": [5.0, 20.0, 4.0, 10.0],
                  "total": [10.0, 20.0, 15.0, 20.0]})
bad = o["total"] != o["qty"] * o["price"]
print(bad.sum(), "row breaks the rule")
o[bad]

### 3 · Duplication
`duplicated()` is True for every row identical to an **earlier** row.

In [ ]:
print(df.duplicated().sum(), "duplicate row")
df[df.duplicated(keep=False)]       # keep=False -> show every copy

`subset=` checks only a key. Here two rows share `id = 2` but have different prices.

In [ ]:
k = pd.DataFrame({"id": [1, 2, 2, 3],
                  "item": ["pen", "cup", "cup", "mug"],
                  "price": [2.0, 5.0, 5.5, 8.0]})
print(k.duplicated().sum(), "exact duplicates")
print(k.duplicated(subset="id").sum(), "repeated id")
print(k["id"].is_unique)

### 4 · Accuracy
`describe()` shows min and max: impossible extremes jump out.

In [ ]:
df[["age", "total"]].describe()

`between(a, b)` tests a range (both ends included). NaN is never in the range, so keep only existing values with `notna()`.

In [ ]:
ok = df["age"].between(0, 110)
print((~ok).sum(), "<- wrong: also counts the NaN")
bad_age = df["age"].notna() & ~ok
print(bad_age.sum(), "<- right: one impossible age")
df[bad_age]

`isin(list)` tests a set of allowed values; `~` reverses the mask.

In [ ]:
allowed = ["shipped", "delivered", "returned", "cancelled"]
not_allowed = ~df["status"].isin(allowed)
df.loc[not_allowed, "status"]

IQR rule: values far outside the middle 50% are **suspicious**, not automatically wrong.

In [ ]:
days = pd.Series([2, 3, 3, 4, 5, 4, 30])
q1, q3 = days.quantile(0.25), days.quantile(0.75)
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(q1, q3, iqr, low, high)
days[(days < low) | (days > high)]

### 5 · The quality report
One named mask per check, then one row per check.

In [ ]:
miss  = df.isna().any(axis=1)
unk   = df["status"] == "unknown"
spell = df["country"] != norm
dup   = df.duplicated()
neg   = df["total"] < 0

report = pd.DataFrame({
    "check": ["missing value", "status unknown", "country not standard",
              "duplicate row", "age outside 0-110", "total below 0"],
    "rows": [miss.sum(), unk.sum(), spell.sum(), dup.sum(), bad_age.sum(), neg.sum()],
})
report["pct"] = (report["rows"] / len(df) * 100).round(1)
report

## Part 2 — Practice (60 minutes)

**The scenario.** The city mobility office from Session 2 received a new export of the March 2019 taxi rides.
Before anyone computes statistics, your manager asks you for a **data quality assessment**:
*what is wrong with this file, where, and how much?*

**Rules**
- **Do not modify `df`.** You only look and count; cleaning comes in the next sessions.
- One empty cell per question. Add cells freely.
- End each answer with a `#` comment that states the result **in words**, e.g. `# 44 rides have no payment`.
- Keep your masks in variables with clear names: you will reuse them in Q10.

**Dataset:** `data/taxis_dirty.csv` — 15 columns: `ride_id` (one id per ride), `pickup`, `dropoff`, `passengers`, `distance`,
`fare`, `tip`, `tolls`, `total`, `color`, `payment`, `pickup_zone`, `dropoff_zone`, `pickup_borough`, `dropoff_borough`.
Real problems plus a few added on purpose: you do not know how many.

In [ ]:
import pandas as pd

# The data folder must sit next to this notebook
df = pd.read_csv("data/taxis_dirty.csv", parse_dates=["pickup", "dropoff"])
df.head()

### ★ Warm-up

**★ · Q1 — First look.**
1. How many rows and columns does the table have?
2. Print the dtype of every column. Which columns hold numbers, which hold text, which hold dates?

**★ · Q2 — Missing cells per column.**
1. How many missing values does each column have?
2. Which column has the most? How many missing cells are there in the whole table?

**★ · Q3 — Share of missing values.**
1. Compute the **percentage** of missing values per column, rounded to 2 decimals.
2. How many rows have **at least one** missing value? *(Hint: `any(axis=1)`)*

### ★★ Look closer

**★★ · Q4 — Missing values in disguise.**
1. Show the `value_counts()` of `payment`, **including** NaN (`dropna=False`).
2. Which value is a placeholder for a missing payment? How many rides have it?
3. How many rides have **no usable payment** in total (NaN + placeholder)?

**★★ · Q5 — One payment, many spellings.**
1. How many distinct values does `payment` have (`nunique()`)? How many after `.str.strip().str.lower()`?
2. How many rides have a payment written in a non-standard way?
   *Careful: NaN is never equal to anything, so keep only the payments that exist (`notna()`).*
3. Look at the `value_counts()` of `pickup_borough`: do you see a spelling problem there too?

**★★ · Q6 — Exact duplicates.**
1. How many rows are exact copies of an earlier row?
2. Show **all** copies of the duplicated rows, sorted by `ride_id` (the first 6 rows are enough).

**★★ · Q7 — The same ride twice?**
`ride_id` should identify one ride.
1. Is `ride_id` unique?
2. How many rows repeat a `ride_id` that already appeared?
3. How many of them are **not** exact duplicates (same id, different values)?

### ★★★ Rules and report

**★★★ · Q8 — Impossible values.**
1. Run `describe()` on `passengers`, `distance`, `fare`, `tip`, `tolls`, `total`. Which min or max looks wrong?
2. Count the rides with a **negative fare**.
3. Count the rides with a number of passengers **outside 1–6**. *(Hint: `between()`)*
4. Count the rides with a **distance of 0**.

**★★★ · Q9 — Do the dates agree?**
1. How many rides **end before they start** (`dropoff` earlier than `pickup`)?
2. The file should contain only March 2019. How many rides were picked up in another month? *(Hint: `.dt.month`)*

**★★★ · Q10 — The quality report.**
Build a DataFrame `report` with one row per check you made in Q3–Q9 (at least 8 checks) and two columns:
`rows` (how many rows fail the check) and `pct` (share of all rows, in %, 2 decimals).
Sort it from the most to the least frequent problem. Which problem is the most common?